<a href="https://colab.research.google.com/github/pratik20034/Pratheek/blob/main-point/dog_breed_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dog Breed Classifier**


### **Explanation:**
- `os`: For file and directory operations.
- `numpy`: Used for numerical computations and array manipulations.
- `random`: For generating random numbers (used for random image selection in prediction).
- `matplotlib.pyplot`: Used for plotting graphs (accuracy/loss plots).
- `seaborn`: Used for visualizing the confusion matrix.
- `PIL.Image`: Used for image loading and preprocessing.
- `sklearn.preprocessing.LabelEncoder`: Encodes labels into numerical format.
- `sklearn.model_selection.train_test_split`: Splits dataset into training and validation sets.
- `sklearn.metrics.classification_report, confusion_matrix`: Used to evaluate the model's performance.
- `tensorflow.keras`: Provides deep learning functionalities such as defining and training a CNN model.
- `Adam`: Optimizer with learning rate adjustments.

---


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam



## **2. Loading Images from Folder**

# **Dog Breed Classifier - Code Explanation**

## **1. Importing Required Libraries**

### **Explanation:**
- `os`: To access and iterate through directories and files.
- `numpy`: For handling images as NumPy arrays.
- `PIL.Image`: To load and resize images.


### **Explanation:**

#### **Looping Through Class Labels (Folders)**

- **`os.listdir(folder)`**: Lists all files and subdirectories inside the given folder.
- **`label.startswith('.')`**: Skips hidden system files like `.DS_Store`, which appear in macOS.
- **`os.path.join(folder, label)`**: Forms the complete path to each class label folder.

#### **Processing Only Directories (Class Labels)**
```python
        if os.path.isdir(label_path):  # Ensure it's a directory
```
- **`os.path.isdir(label_path)`**: Checks if `label_path` is actually a folder and **not a file**.
- **Prevents accidental inclusion of unwanted files**.

#### **Looping Through Images in Each Folder**
```python
            for filename in os.listdir(label_path):
                if filename.startswith('.'):  # Ignore hidden files
                    continue
```
- **`os.listdir(label_path)`**: Lists all images inside the class label folder.
- **`filename.startswith('.')`**: Ignores hidden files (e.g., `.DS_Store` and other system-generated files).

#### **Loading and Preprocessing Images**

- **`Image.open(img_path)`**: Opens the image file.
- **`img.resize(image_size)`**: Resizes the image to a fixed size `(255, 255)`.
- **`np.array(img)`**: Converts the image to a NumPy array for further processing.
- **`images.append(img)`**: Stores the image data in the `images` list.
- **`labels.append(label)`**: Stores the corresponding class label.

#### **Returning Processed Images & Labels**
```python
    return np.array(images), np.array(labels)
```
- **Converts `images` and `labels` lists into NumPy arrays** for easy manipulation in machine learning models.
- **Returns both `images` and `labels`** for training the model.

---


In [9]:

# Helper function to load images
def load_images_from_folder(folder, image_size=(255, 255)):
    images = []
    labels = []

    for label in os.listdir(folder):
        if label.startswith('.'):  # Ignore hidden files (like .DS_Store)
            continue

        label_path = os.path.join(folder, label)
        if os.path.isdir(label_path):
            for filename in os.listdir(label_path):
                if filename.startswith('.'):  # Ignore hidden files
                    continue

                img_path = os.path.join(label_path, filename)
                img = Image.open(img_path)
                img = img.resize(image_size)
                img = np.array(img)
                images.append(img)
                labels.append(label)

    return np.array(images), np.array(labels)




## **3. Preprocessing Data**

### **Explanation:**
- Loads images and labels from the dataset.
- Normalizes pixel values between `0` and `1`.
- Encodes labels into numerical format.
- Converts labels into **one-hot encoding** format.
- Splits the dataset into **training (80%)** and **validation (20%)** sets.


In [6]:
# Load data
data_dir = '/content/drive/MyDrive/CNN Dog/dataset'
image_size = (255, 255)
X, y = load_images_from_folder(data_dir, image_size)


In [10]:
# Encode labels to integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


In [11]:
# Convert labels to one-hot encoding
y = to_categorical(y)


In [12]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


---

## **4. Defining CNN Model**

<img src= "https://vitalflux.com/wp-content/uploads/2021/11/VGG16-CNN-Architecture.png">

### **Explanation:**
- **Conv2D(32, (3,3))**: Extracts features using a **32-filter convolution layer**.
- **MaxPooling2D(2,2)**: Reduces the image size to prevent overfitting.
- **Conv2D(64, (3,3))**: Adds a deeper convolution layer with **64 filters**.
- **Conv2D(128, (3,3))**: Further feature extraction using **128 filters**.
- **Flatten()**: Converts the multi-dimensional feature map into a **1D vector**.
- **Dense(512, activation='relu')**: Fully connected layer with 512 neurons.
- **Dropout(0.5)**: Prevents overfitting by randomly disabling 50% of neurons.
- **Dense(output_classes, activation='softmax')**: Output layer using **softmax** for classification.

<img src= "https://www.researchgate.net/publication/351361078/figure/fig2/AS:1021239741120515@1620493936749/Basic-CNN-architecture-for-image-classification.png">



In [13]:
# Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(255, 255, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(len(label_encoder.classes_), activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
# Compile the model with Adam optimizer (learning rate = 0.0001)
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.0001),
    metrics=['accuracy']
)

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val)
)

Epoch 1/20
12/25 ━━━━━━━━━━━━━━━━━━━━ 1:20 6s/step - accuracy: 0.9717 - loss: 0.0806

---

## **5. Compiling and Training the Model**

### **Explanation:**
- Uses `Adam` optimizer with a **learning rate of 0.0001** for better convergence.
- Uses `categorical_crossentropy` since it’s a multi-class classification problem.
- Tracks **accuracy** as the performance metric.

---

## **6. Training the Model**

### **Explanation:**
- **Epochs: 30** → The model will train for 30 cycles.
- **Batch size: 32** → Trains using 32 images at a time.
- Uses **validation data** to track performance.

---

## **7. Model Evaluation**

### **Explanation:**
- Evaluates the model on validation data and prints the **accuracy**.

---



In [ ]:
# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(X_val, y_val)
print(f'Validation accuracy: {val_accuracy:.4f}')

In [ ]:
# Save the model
model.save('dog_breed_classifier_model.h5')

In [ ]:
# Generate classification report and confusion matrix
y_val_pred = model.predict(X_val)
y_val_pred_classes = np.argmax(y_val_pred, axis=1)
y_val_true_classes = np.argmax(y_val, axis=1)


## **8. Generating Classification Report & Confusion Matrix**

### **Explanation:**
- Converts predictions into class labels.
- Generates a **classification report** and saves it.
- Generates a **confusion matrix** for visual analysis.

---



In [ ]:
class_report = classification_report(y_val_true_classes, y_val_pred_classes, target_names=label_encoder.classes_)
conf_matrix = confusion_matrix(y_val_true_classes, y_val_pred_classes)

In [ ]:
# Save the classification report to a file
with open("classification_report.txt", "w") as f:
    f.write(class_report)

print("Classification Report:\n", class_report)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

## **9. Plotting Accuracy & Loss**

### **Explanation:**
- Plots **training vs validation accuracy**.
- Plots **training vs validation loss**.
- Helps in understanding **overfitting** or **underfitting**.

---

## **Conclusion**
This script successfully trains a **CNN-based Dog Breed Classifier** with performance evaluation, classification metrics, and visualization of accuracy/loss trends.

In [ ]:
# Plot accuracy
# Plot Accuracy and Loss curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')

In [ ]:
# Plot loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')

plt.show()